In [0]:
dbutils.widgets.removeAll()

from pyspark.sql.functions import col, trim, upper, regexp_replace, current_timestamp, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_smartdata_final")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlssmartdata2023")
dbutils.widgets.text("fileName", "electric_vehicles_spec_2025.csv.csv")
dbutils.widgets.text("targetTable", "ev_specs_2025_bronze")
dbutils.widgets.dropdown("writeMode", "overwrite", ["overwrite", "append"])

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")
file_name = dbutils.widgets.get("fileName")
target_table = dbutils.widgets.get("targetTable")
write_mode = dbutils.widgets.get("writeMode")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/{file_name}"
tabla_destino = f"{catalogo}.{esquema}.{target_table}"

print(f"Ruta origen: {ruta}")
print(f"Tabla destino: {tabla_destino}")

Ruta origen: abfss://raw@adlssmartdata2023.dfs.core.windows.net/electric_vehicles_spec_2025.csv.csv
Tabla destino: catalog_smartdata_final.bronze.ev_specs_2025_bronze


In [0]:
ev_specs_schema = StructType([
    StructField("brand", StringType(), True),
    StructField("model", StringType(), True),
    StructField("top_speed_kmh", IntegerType(), True),
    StructField("battery_capacity_kWh", DoubleType(), True),
    StructField("battery_type", StringType(), True),
    StructField("number_of_cells", IntegerType(), True),
    StructField("torque_nm", IntegerType(), True),
    StructField("efficiency_wh_per_km", DoubleType(), True),
    StructField("range_km", IntegerType(), True),
    StructField("acceleration_0_100_s", DoubleType(), True),
    StructField("fast_charging_power_kw_dc", DoubleType(), True),
    StructField("fast_charge_port", StringType(), True),
    StructField("towing_capacity_kg", StringType(), True),
    StructField("cargo_volume_l", StringType(), True),
    StructField("seats", IntegerType(), True),
    StructField("drivetrain", StringType(), True),
    StructField("segment", StringType(), True),
    StructField("length_mm", IntegerType(), True),
    StructField("width_mm", IntegerType(), True),
    StructField("height_mm", IntegerType(), True),
    StructField("car_body_type", StringType(), True),
    StructField("source_url", StringType(), True)
 ])

In [0]:
df_specs = spark.read\
    .option("header", True)\
    .schema(ev_specs_schema)\
    .csv(ruta)

normalize_text = lambda c: upper(trim(regexp_replace(col(c), r"\\s+", " ")))

df_specs_bronze = df_specs.select(
    normalize_text("brand").alias("brand"),
    normalize_text("model").alias("model"),
    col("top_speed_kmh"),
    col("battery_capacity_kWh").alias("battery_capacity_kwh"),
    col("battery_type"),
    col("number_of_cells"),
    col("torque_nm"),
    col("efficiency_wh_per_km"),
    col("range_km"),
    col("acceleration_0_100_s"),
    col("fast_charging_power_kw_dc"),
    col("fast_charge_port"),
    col("towing_capacity_kg"),
    col("cargo_volume_l"),
    col("seats"),
    col("drivetrain"),
    col("segment"),
    col("length_mm"),
    col("width_mm"),
    col("height_mm"),
    col("car_body_type"),
    col("source_url"),
    lit(file_name).alias("source_file"),
    current_timestamp().alias("ingestion_ts")
)

In [0]:
df_specs_bronze.write.mode(write_mode).insertInto(tabla_destino)

print(f"Registros escritos: {df_specs_bronze.count()}")
display(df_specs_bronze.limit(10))

Registros escritos: 478


brand,model,top_speed_kmh,battery_capacity_kwh,battery_type,number_of_cells,torque_nm,efficiency_wh_per_km,range_km,acceleration_0_100_s,fast_charging_power_kw_dc,fast_charge_port,towing_capacity_kg,cargo_volume_l,seats,drivetrain,segment,length_mm,width_mm,height_mm,car_body_type,source_url,source_file,ingestion_ts
ABARTH,500E CONVERTIBLE,155,37.8,Lithium-ion,192,235,156.0,225,7.0,67.0,CCS,0,185,4,FWD,B - Compact,3673,1683,1518,Hatchback,https://ev-database.org/car/1904/Abarth-500e-Convertible,electric_vehicles_spec_2025.csv.csv,2026-03-15T23:40:14.46326Z
ABARTH,500E HATCHBACK,155,37.8,Lithium-ion,192,235,149.0,225,7.0,67.0,CCS,0,185,4,FWD,B - Compact,3673,1683,1518,Hatchback,https://ev-database.org/car/1903/Abarth-500e-Hatchback,electric_vehicles_spec_2025.csv.csv,2026-03-15T23:40:14.46326Z
ABARTH,600E SCORPIONISSIMA,200,50.8,Lithium-ion,102,345,158.0,280,5.9,79.0,CCS,0,360,5,FWD,JB - Compact,4187,1779,1557,SUV,https://ev-database.org/car/3057/Abarth-600e-Scorpionissima,electric_vehicles_spec_2025.csv.csv,2026-03-15T23:40:14.46326Z
ABARTH,600E TURISMO,200,50.8,Lithium-ion,102,345,158.0,280,6.2,79.0,CCS,0,360,5,FWD,JB - Compact,4187,1779,1557,SUV,https://ev-database.org/car/3056/Abarth-600e-Turismo,electric_vehicles_spec_2025.csv.csv,2026-03-15T23:40:14.46326Z
AIWAYS,U5,150,60.0,Lithium-ion,null,310,156.0,315,7.5,78.0,CCS,null,496,5,FWD,JC - Medium,4680,1865,1700,SUV,https://ev-database.org/car/1678/Aiways-U5,electric_vehicles_spec_2025.csv.csv,2026-03-15T23:40:14.46326Z
AIWAYS,U6,160,60.0,Lithium-ion,null,315,150.0,350,7.0,78.0,CCS,null,472,5,FWD,JC - Medium,4805,1880,1641,SUV,https://ev-database.org/car/1766/Aiways-U6,electric_vehicles_spec_2025.csv.csv,2026-03-15T23:40:14.46326Z
ALFA,ROMEO JUNIOR ELETTRICA 54 KWH,150,50.8,Lithium-ion,102,260,128.0,320,9.0,85.0,CCS,0,400,5,FWD,JB - Compact,4173,1781,1532,SUV,https://ev-database.org/car/2184/Alfa-Romeo-Junior-Elettrica-54-kWh,electric_vehicles_spec_2025.csv.csv,2026-03-15T23:40:14.46326Z
ALFA,ROMEO JUNIOR ELETTRICA 54 KWH VELOCE,200,50.8,Lithium-ion,102,345,164.0,310,6.0,85.0,CCS,0,400,5,FWD,JB - Compact,4173,1781,1505,SUV,https://ev-database.org/car/2185/Alfa-Romeo-Junior-Elettrica-54-kWh-Veloce,electric_vehicles_spec_2025.csv.csv,2026-03-15T23:40:14.46326Z
ALPINE,A290 ELECTRIC 180 HP,160,52.0,Lithium-ion,184,285,138.0,310,7.4,70.0,CCS,500,326,5,FWD,B - Compact,3997,1823,1512,Hatchback,https://ev-database.org/car/2268/Alpine-A290-Electric-180-hp,electric_vehicles_spec_2025.csv.csv,2026-03-15T23:40:14.46326Z
ALPINE,A290 ELECTRIC 220 HP,170,52.0,Lithium-ion,184,300,144.0,305,6.4,70.0,CCS,500,326,5,FWD,B - Compact,3997,1823,1512,Hatchback,https://ev-database.org/car/2269/Alpine-A290-Electric-220-hp,electric_vehicles_spec_2025.csv.csv,2026-03-15T23:40:14.46326Z
